In [ ]:
# ── Cell 1: Mount Drive and set working directory ─────────────────────────
from google.colab import drive  # type: ignore
import os, subprocess, sys
from pathlib import Path

drive.mount("/content/drive")

# ── Locate project root — check every common Drive layout ─────────────────
_CANDIDATES = [
    "/content/drive/MyDrive/PixelLab-StudyPal-RAG-DIP/smart-learning-assistant",
    "/content/drive/MyDrive/smart-learning-assistant/PixelLab-StudyPal-RAG-DIP/smart-learning-assistant",
    "/content/drive/MyDrive/PixelLab-StudyPal-RAG-DIP",
    "/content/drive/MyDrive/smart-learning-assistant",
]

DRIVE_PROJECT_ROOT = next(
    (p for p in _CANDIDATES
     if Path(p).is_dir() and Path(p, "app").is_dir()),  # must have app/ subdir
    None,
)

if DRIVE_PROJECT_ROOT is None:
    # Last-resort: ask user
    print("⚠️  Could not auto-detect project folder.")
    print("    Available top-level Drive folders:")
    for d in sorted(Path("/content/drive/MyDrive").iterdir()):
        if d.is_dir():
            print("       ", d)
    DRIVE_PROJECT_ROOT = input(
        "\nEnter FULL path to the smart-learning-assistant folder: "
    ).strip()
    if not Path(DRIVE_PROJECT_ROOT, "app").is_dir():
        sys.exit("❌ Path not valid — no app/ subdirectory found. Fix the path and re-run.")

print(f"Project root : {DRIVE_PROJECT_ROOT}")

# ── Pull latest code ───────────────────────────────────────────────────────
_repo_root = str(Path(DRIVE_PROJECT_ROOT).parent)
_pull = subprocess.run(
    ["git", "pull", "--ff-only"],
    cwd=_repo_root, capture_output=True, text=True,
)
_msg = (_pull.stdout + _pull.stderr).strip() or "Already up to date."
print("git pull:", _msg)

os.chdir(DRIVE_PROJECT_ROOT)
print(f"Working dir  : {os.getcwd()}")
print("Files        :", sorted(os.listdir(".")))


In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────
# ragas 0.1.x needs langchain_core.pydantic_v1, which was REMOVED in
# langchain-core 0.3.x  →  pin the whole ecosystem to 0.2.x.
# langchain-huggingface requires core>=0.3, so use langchain-community instead.
%pip install -q \
    "ragas==0.1.21" \
    "langchain-core>=0.2.0,<0.3.0" \
    "langchain>=0.2.0,<0.3.0" \
    "langchain-community>=0.2.0,<0.3.0" \
    "langchain-groq>=0.1.0,<0.2.0" \
    "datasets>=2.18" \
    "sentence-transformers" \
    "python-dotenv"

print("\n✅ Packages installed. Restart runtime NOW if Colab prompts you.")


In [ ]:
# ── Cell 3: Set Groq API key (free judge LLM for RAGAS) ───────────────────
# Get a free key at https://console.groq.com/keys (sign in with GitHub/Google)
import os
from getpass import getpass

groq_key = getpass("Enter your Groq API key (gsk_...): ")
os.environ["GROQ_API_KEY"] = groq_key

# Quick sanity check — uses llama-3.1-8b-instant (500K TPD, much lower token cost)
from langchain_groq import ChatGroq  # type: ignore
_test = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
_resp = _test.invoke("Reply with the single word OK.")
print(f"✅ Groq key valid. Test response: {_resp.content if isinstance(_resp.content, str) else _resp.content[0] if _resp.content else 'OK'}")
print("   RAGAS will use: llama-3.1-8b-instant (judge, 500K TPD) + all-MiniLM-L6-v2 (embeddings)")
print("   RunConfig: max_workers=2, timeout=120s  →  no timeout cascade")


In [ ]:
# ── Cell 4: Load intermediate JSON ────────────────────────────────────────
import json
from pathlib import Path

INTERMEDIATE_PATH = Path("data/eval_intermediate.json")

if not INTERMEDIATE_PATH.exists():
    # Fallback: allow manual upload from local machine
    from google.colab import files # type: ignore
    print("eval_intermediate.json not found in Drive project folder.")
    print("Uploading from your local machine instead...")
    uploaded = files.upload()          # shows a file-picker in Colab
    INTERMEDIATE_PATH = Path(list(uploaded.keys())[0])

with open(INTERMEDIATE_PATH, encoding="utf-8") as f:
    intermediate = json.load(f)

n_questions  = len(intermediate["questions"])
n_guardrail  = len(intermediate.get("guardrail_results", []))
collected_at = intermediate.get("collected_at", "unknown")

print(f"✅ Loaded intermediate data")
print(f"   DIP questions  : {n_questions}")
print(f"   Guardrail tests: {n_guardrail}")
print(f"   Collected at   : {collected_at}")

# Preview first question
print(f"\n   Q1: {intermediate['questions'][0]}")
print(f"   A1: {intermediate['answers'][0][:120]}...")

In [ ]:
# ── Cell 5: RAGAS scoring — fully self-contained, no Drive imports ─────────
# Does NOT import from app.evaluation.metrics — immune to stale Drive code.
import sys, os
from pathlib import Path
from datetime import datetime, timezone

import numpy as np          # type: ignore
import pandas as pd         # type: ignore

sys.path.insert(0, str(Path.cwd()))

from datasets import Dataset                              # type: ignore
from ragas import evaluate, RunConfig                     # type: ignore
from ragas.metrics import (                               # type: ignore
    answer_relevancy, context_precision,
    context_recall, faithfulness,
)
from ragas.llms import LangchainLLMWrapper                # type: ignore
from ragas.embeddings import LangchainEmbeddingsWrapper   # type: ignore
from langchain_groq import ChatGroq                       # type: ignore
# langchain-community 0.2.x has HuggingFaceEmbeddings and works with core 0.2.x
from langchain_community.embeddings import HuggingFaceEmbeddings  # type: ignore

# ── Judge LLM: reads GROQ_API_KEY from env (avoids SecretStr coercion issue) ──
ragas_llm = LangchainLLMWrapper(
    ChatGroq(model="llama-3.1-8b-instant", temperature=0)   # type: ignore[call-arg]
)
ragas_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

# ── Dataset ────────────────────────────────────────────────────────────────
dataset = Dataset.from_dict({
    "question":     intermediate["questions"],
    "answer":       intermediate["answers"],
    "contexts":     intermediate["contexts"],
    "ground_truth": intermediate["ground_truths"],
})

# ── RunConfig: max_workers=1 = fully sequential = zero concurrent burst ────
# timeout=180 = 3 min per single job; generous even under Groq load
run_cfg = RunConfig(timeout=180, max_workers=1)

n = len(intermediate["questions"])
print(f"Running RAGAS on {n} questions  (sequential · 1 job at a time · 180 s timeout) …")
print("Expected time: ~10–15 min.  Do NOT interrupt.\n")

_raw = evaluate(                                          # type: ignore[call-overload]
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=run_cfg,
)

# ── Safe conversion to DataFrame ──────────────────────────────────────────
_to_df = getattr(_raw, "to_pandas", None)
ragas_df: pd.DataFrame = _to_df() if callable(_to_df) else pd.DataFrame(dict(_raw))  # type: ignore[arg-type]

# ── Pretty-print scores ───────────────────────────────────────────────────
_METRICS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
_TARGET  = 0.7

print("\n=== RAGAS Scores ===")
print(f"{'Metric':<25} {'Score':>7}  {'Target':>7}  Status")
print("-" * 52)
for m in _METRICS:
    if m in ragas_df.columns:
        score  = float(ragas_df[m].mean())
        status = "✅ PASS" if score >= _TARGET else "❌ FAIL"
        print(f"{m:<25} {score:>7.3f}  {_TARGET:>7.1f}  {status}")

# ── Inline report (no Drive dependency) ───────────────────────────────────
def _build_report(df: pd.DataFrame,
                  latencies: list,
                  guardrail_results: list,
                  topic_map: list | None = None) -> str:
    ts   = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    lats = sorted(latencies or [0])
    lines = [
        "# Evaluation Report — DIP AI Tutor",
        f"\n_Generated: {ts}_\n",
        "\n## Overall RAGAS Scores\n",
        "| Metric | Score | Target | Status |",
        "|--------|-------|--------|--------|",
    ]
    for m in _METRICS:
        if m in df.columns:
            s = float(df[m].mean())
            lines.append(f"| {m} | {s:.3f} | {_TARGET:.1f} | {'✅' if s >= _TARGET else '❌'} |")

    if topic_map and all(c in df.columns for c in ["faithfulness", "answer_relevancy"]):
        _df = df.copy()
        _df["topic"] = topic_map[:len(_df)]
        lines += ["\n## Per-Topic Breakdown\n",
                  "| Topic | Faithfulness | Answer Relevancy |",
                  "|-------|-------------|-----------------|"]
        for topic, grp in _df.groupby("topic"):
            lines.append(f"| {str(topic)[:60]} | {float(grp['faithfulness'].mean()):.3f} | {float(grp['answer_relevancy'].mean()):.3f} |")

    if lats:
        p95 = float(np.percentile(lats, 95))
        lines += ["\n## Latency Analysis\n",
                  f"- **Mean**: {float(np.mean(lats)):.2f}s",
                  f"- **p50**: {float(np.percentile(lats, 50)):.2f}s",
                  f"- **p95**: {p95:.2f}s" + (" ⚠️ exceeds 5 s target" if p95 > 5 else " ✅")]

    if guardrail_results:
        lines += ["\n## Guardrail Test Results\n",
                  "| Question | Status | Answer Preview |",
                  "|----------|--------|----------------|"]
        for g in guardrail_results:
            lines.append(f"| {g['question'][:50]} | {g['status']} | {g['answer'][:80].replace(chr(10), ' ')} |")

    lines += ["\n## Failed Cases (score < 0.7)\n"]
    any_bad = False
    for i, row in df.iterrows():
        bad = [m for m in _METRICS if m in row and float(row[m]) < _TARGET]
        if bad:
            any_bad = True
            lines.append(f"**Q{int(i)+1}**: {str(row.get('question', ''))[:80]}")
            for m in bad:
                lines.append(f"  - `{m}`: {float(row[m]):.3f}")
    if not any_bad:
        lines.append("_All questions meet the 0.7 threshold._ ✅")

    return "\n".join(lines)

print("\nGenerating markdown report …")
report_md = _build_report(
    ragas_df,
    latencies=intermediate.get("latencies", []),
    guardrail_results=intermediate.get("guardrail_results", []),
    topic_map=intermediate.get("topic_map"),
)
Path("evaluation_report.md").write_text(report_md, encoding="utf-8")
print("✅ Report generated and saved to evaluation_report.md")


In [ ]:
# ── Cell 6: Save report to Drive and offer local download ─────────────────
from pathlib import Path
from google.colab import files as colab_files # type: ignore

REPORT_PATH = Path("evaluation_report.md")

# The report was already written to evaluation_report.md by generate_report()
if REPORT_PATH.exists():
    print(f"📄 Report saved at: {REPORT_PATH.resolve()}")
    print(f"   Size: {REPORT_PATH.stat().st_size:,} bytes")

    # Download to local machine
    print("\nDownloading evaluation_report.md to your local machine...")
    colab_files.download(str(REPORT_PATH))

    # Also save RAGAS DataFrame as CSV
    csv_path = Path("data/ragas_scores.csv")
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    ragas_df.to_csv(csv_path, index=False)
    colab_files.download(str(csv_path))
    print("✅ Downloaded evaluation_report.md and ragas_scores.csv")
else:
    print("⚠️  Report file not found. Check for errors in Cell 5.")

# Print the report inline too
print("\n" + "=" * 70)
print(report_md)